In [1]:
from allel.stats.roh import roh_mhmm
import hmmlearn
print(hmmlearn.__version__)
import allel
import numpy as np
import pandas as pd


0.3.3


In [2]:
import scipy, hmmlearn, allel
print(scipy.__version__)
print(hmmlearn.__version__)
print(allel.__version__)

1.17.0
0.3.3
1.3.13


In [3]:
import malariagen_data
af1 = malariagen_data.Af1()

In [4]:
sample_sets=[
        "1188-VO-NIANG-NIEL-SN-2304-VMF00259",
        "1330-VO-GN-LAMA-VMF00250",
        "1315-VO-NG-OMITOLA-OMOL-NG-2008",
        "1281-VO-CM-CHRISTOPHE-VMF00227",
        "AG1000G-KE"]

In [5]:

gt_da = af1.snp_genotypes(
    region="2RL",
    sample_sets=sample_sets,
    sample_query='taxon == "funestus" and sex_call == "F"',
    site_mask="funestus",
    field="GT",
)

In [6]:
pos = af1.snp_variants(
    region="2RL",
    site_mask="funestus",
)["variant_position"].values

In [8]:
contig_size = af1.genome_sequence("2RL").shape[0]
MIN_ROH = 100_000
roh_records = []

In [7]:
TEST_IDX = 1
gti = gt_da[:, TEST_IDX, :].compute()
gti.shape

(58829365, 2)

In [10]:
roh = roh_mhmm(
    gti,
    pos,
    phet_roh=0.0015,
    phet_nonroh=(0.0025, 0.01, 0.015),
    transition=1e-6,
    min_roh=MIN_ROH,
    contig_size=contig_size,
)

roh

(         start       stop   length  is_marginal
 165   24028784   24148473   119690        False
 176   25887687   25994086   106400        False
 328   44889018   45040954   151937        False
 377   55323355   55463882   140528        False
 379   56476906   56607683   130778        False
 380   56757091   57779344  1022254        False
 381   57782804   58093612   310809        False
 386   60580226   60698365   118140        False
 689  102249559  102380917   131359        False
 690  102603661  102883511   279851         True,
 np.float64(0.024413494208999145))

In [39]:
sample_sets=[
        "1188-VO-NIANG-NIEL-SN-2304-VMF00259",
        "1330-VO-GN-LAMA-VMF00250",
        "1315-VO-NG-OMITOLA-OMOL-NG-2008",
        "1281-VO-CM-CHRISTOPHE-VMF00227",
        "AG1000G-KE"]

WINDOW_BP = 50_000        # window size (50 kb)
MAX_HET_PER_WINDOW = 1   # allow at most 1 heterozygous SNP per window
MIN_ROH_BP = 100_000     # minimum ROH length (100 kb)
SITE_MASK = "funestus"
CHROM = "2RL"


In [40]:
gt_da = af1.snp_genotypes(
    region=CHROM,
    sample_sets=sample_sets,
    sample_query='taxon == "funestus" and sex_call == "F"',
    site_mask=SITE_MASK,
    field="GT",
)


Access SNP genotypes: ⠋ (0:00:00.00)

In [41]:
gt_da.shape


(58829365, 618, 2)

In [42]:
pos = af1.snp_variants(
    region=CHROM,
    site_mask=SITE_MASK,
)["variant_position"].values

assert gt_da.shape[0] == len(pos)


In [43]:
meta = af1.sample_metadata(
    sample_sets=sample_sets,
    sample_query='taxon == "funestus" and sex_call == "F"',
)[["sample_id", "country"]].reset_index(drop=True)

sample_ids = meta["sample_id"].values


In [45]:
def call_roh_sliding_window(pos, is_het,
                            window_bp=50_000,
                            max_het=1,
                            min_roh_bp=100_000):
    """
    PLINK-style ROH using sliding windows in bp.
    """

    # define windows
    win_starts = np.arange(pos[0], pos[-1], window_bp)
    win_ends = win_starts + window_bp

    roh_windows = []

    for ws, we in zip(win_starts, win_ends):
        idx = (pos >= ws) & (pos < we)
        if idx.sum() == 0:
            continue

        het_count = is_het[idx].sum()
        if het_count <= max_het:
            roh_windows.append((ws, we))

    # merge consecutive ROH windows
    roh_segments = []
    if roh_windows:
        cur_start, cur_end = roh_windows[0]

        for ws, we in roh_windows[1:]:
            if ws == cur_end:
                cur_end = we
            else:
                length = cur_end - cur_start
                if length >= min_roh_bp:
                    roh_segments.append((cur_start, cur_end, length))
                cur_start, cur_end = ws, we

        # last segment
        length = cur_end - cur_start
        if length >= min_roh_bp:
            roh_segments.append((cur_start, cur_end, length))

    return pd.DataFrame(
        roh_segments,
        columns=["start", "stop", "length"]
    )


In [46]:
TEST_IDX = 1

gti = allel.GenotypeArray(
    gt_da[:, TEST_IDX:TEST_IDX+1, :].compute()
)

is_het = gti.is_het().flatten()

roh_test = call_roh_sliding_window(
    pos=pos,
    is_het=is_het,
    window_bp=WINDOW_BP,
    max_het=MAX_HET_PER_WINDOW,
    min_roh_bp=MIN_ROH_BP,
)

roh_test


,start,stop,length
0,57126585,57376585,250000
1,57426585,57526585,100000


In [36]:
def call_roh_from_het_gaps(pos, is_het, min_roh):
    """
    Define ROH as long genomic intervals between heterozygous SNPs.
    """
    het_pos = pos[is_het]

    roh_list = []

    # start of region
    prev = pos[0]

    for hp in het_pos:
        length = hp - prev
        if length >= min_roh:
            roh_list.append((prev, hp, length))
        prev = hp

    # tail
    length = pos[-1] - prev
    if length >= min_roh:
        roh_list.append((prev, pos[-1], length))

    return pd.DataFrame(roh_list, columns=["start", "stop", "length"])


In [37]:
roh_i = call_roh_from_hets(
    pos=pos,
    is_het=is_het,
    min_roh=MIN_ROH,
)
roh_i


,start,stop,length


In [38]:
het_pos = pos[is_het]

# distances between consecutive heterozygous SNPs
het_gaps = np.diff(het_pos)

het_gaps.max(), np.percentile(het_gaps, [90, 95, 99, 99.9])


(np.int32(59592), array([ 301.  ,  448.  , 1058.  , 4732.36]))